# algo10：局部保持投影 LPP（高精度真实增强版）

**场景：材料设计**  
**UCI 数据集：Superconductivity Data**。

增强点：
1. 真实 UCI 超导材料数据；
2. 样本数和特征精度提高；
3. 量子 LPP 在特征空间求投影方向，QUBO 变量数可控，所以直接使用 8 档高精度编码；
4. 新增 LPP 目标误差与局部结构指标。


In [ ]:
# ==========================================================
# 通用配置：UCI 数据 + 传统/手动/量子真机对比
# ==========================================================
import os, time, zipfile, warnings, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
warnings.filterwarnings("ignore")

from scipy.spatial.distance import cdist
from scipy.linalg import eigh as scipy_eigh
from scipy.special import expit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import SpectralEmbedding, LocallyLinearEmbedding, trustworthiness
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score, v_measure_score
from sklearn.metrics import silhouette_score, mean_squared_error, mean_absolute_error, f1_score, precision_score, recall_score, r2_score
import sklearn.metrics.cluster

# 量子真机/CIM：必须调用 kaiwu；没有本地模拟兜底
import kaiwu as kw

def init_kaiwu_license_once():
    """优先复用本地 license，避免每次运行都重新下载。"""
    try:
        kw.license.ensure_license()
        print('✅ Kaiwu license 已存在，直接使用本地 license。')
    except Exception as e:
        print('⚠️ 本地 license 不存在或不可用，尝试重新初始化。')
        print('原始信息:', repr(e))
        kw.license.init(user_id=os.getenv('KAIWU_USER_ID', '129788932442292226'),
                        sdk_code=os.getenv('KAIWU_SDK_CODE', 'z8zz2hephR3iY1ZiypMphxMV9Tn7fR'))
        print('✅ Kaiwu license 初始化完成。')

init_kaiwu_license_once()
kw.common.CheckpointManager.save_dir = '/tmp'

DATA_DIR = Path('uci_data_cache')
RESULT_DIR = Path('results')
DATA_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)

np.random.seed(42)
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print('kaiwu initialized; UCI cache:', DATA_DIR.resolve())


# ==========================================================
# UCI 下载工具：官方链接 + HTTP/HTTPS 备份 + verify=False 解决 SSL 失败
# ==========================================================
def fetch_file(urls, filename, min_bytes=128):
    path = DATA_DIR / filename
    if path.exists() and path.stat().st_size >= min_bytes:
        print(f"使用本地缓存: {path}")
        return path
    last_error = None
    for url in urls:
        try:
            print(f"尝试下载: {url}")
            with requests.get(url, stream=True, verify=False, timeout=90) as r:
                r.raise_for_status()
                with open(path, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=1024*1024):
                        if chunk:
                            f.write(chunk)
            if path.exists() and path.stat().st_size >= min_bytes:
                print(f"下载成功: {path} ({path.stat().st_size/1024:.1f} KB)")
                return path
        except Exception as e:
            last_error = e
            print(f"下载失败，换下一个源: {type(e).__name__}: {e}")
            if path.exists() and path.stat().st_size < min_bytes:
                try: path.unlink()
                except Exception: pass
    raise RuntimeError(f"所有下载源均失败。最后错误: {last_error}")

def first_zip_member(zip_path, keywords):
    keywords = [k.lower() for k in keywords]
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
        for name in names:
            low = name.lower()
            if all(k in low for k in keywords) and not low.endswith('/'):
                return name
    raise FileNotFoundError(f"zip 中找不到包含 {keywords} 的文件: {zip_path}")

def balanced_sample(X, y, n_per_class=30, random_state=42):
    rng = np.random.default_rng(random_state)
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)
    idx_all = []
    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        take = min(n_per_class, len(idx))
        idx_all.extend(rng.choice(idx, size=take, replace=False))
    idx_all = np.array(idx_all)
    rng.shuffle(idx_all)
    return X[idx_all], y[idx_all]

def purity_score(y_true, y_pred):
    cm = sklearn.metrics.cluster.contingency_matrix(y_true, y_pred)
    return np.sum(np.amax(cm, axis=0)) / np.sum(cm)

def safe_kmeans_labels(Z, n_clusters=2, random_state=42):
    Z = np.asarray(Z, dtype=float)
    if Z.ndim == 1:
        Z = Z.reshape(-1, 1)
    return KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10).fit_predict(Z)

def knn_preservation(X_high, X_low, k=8):
    X_high = np.asarray(X_high); X_low = np.asarray(X_low)
    k = min(k, len(X_high)-1)
    Dh = cdist(X_high, X_high)
    Dl = cdist(X_low, X_low)
    Nh = np.argsort(Dh, axis=1)[:, 1:k+1]
    Nl = np.argsort(Dl, axis=1)[:, 1:k+1]
    return float(np.mean([len(set(a).intersection(set(b))) / k for a, b in zip(Nh, Nl)]))

def embedding_metrics(y_true, X_high, Z, name, runtime, k=8):
    n_clusters = len(np.unique(y_true))
    pred = safe_kmeans_labels(Z, n_clusters=n_clusters)
    k_eff = min(k, len(y_true)-2)
    sil = silhouette_score(Z, pred) if len(np.unique(pred)) > 1 else np.nan
    return {
        'Method': name,
        'NMI': normalized_mutual_info_score(y_true, pred),
        'ARI': adjusted_rand_score(y_true, pred),
        'V-measure': v_measure_score(y_true, pred),
        'Purity': purity_score(y_true, pred),
        'Silhouette': sil,
        'Trustworthiness': trustworthiness(X_high, Z, n_neighbors=max(2, k_eff)),
        'KNN-Preservation': knn_preservation(X_high, Z, k=max(2, k_eff)),
        'Time(s)': runtime,
    }

def matrix_metrics(R_true, R_pred, missing_mask, name, runtime):
    diff = R_pred[missing_mask] - R_true[missing_mask]
    rmse = float(np.sqrt(np.mean(diff**2)))
    mae = float(np.mean(np.abs(diff)))
    rel_frob = float(np.linalg.norm((R_pred - R_true)[missing_mask]) / (np.linalg.norm(R_true[missing_mask]) + 1e-12))
    try:
        r2 = float(r2_score(R_true[missing_mask].ravel(), R_pred[missing_mask].ravel()))
    except Exception:
        r2 = np.nan
    return {'Method': name, 'RMSE': rmse, 'MAE': mae, 'RelativeError': rel_frob, 'R2': r2, 'Time(s)': runtime}

def add_bar_labels(ax, fmt='{:.3f}', rotation=0):
    for p in ax.patches:
        h = p.get_height()
        if np.isfinite(h):
            ax.annotate(fmt.format(h), (p.get_x()+p.get_width()/2, h), ha='center', va='bottom', fontsize=8, rotation=rotation)

# ==========================================================
# QUBO / Ising / kaiwu CIM 工具
# ==========================================================
def scale_to_qubo_precision(Q, bit_width=8):
    Q = np.asarray(Q, dtype=float)
    Q = (Q + Q.T) / 2
    q_min, q_max = np.min(Q), np.max(Q)
    if abs(q_max - q_min) < 1e-12:
        Q_scaled = np.zeros_like(Q)
    else:
        Q_scaled = ((Q - q_min) / (q_max - q_min)) * 255 - 128
    Q_clipped = np.clip(np.round(Q_scaled), -128, 127)
    return kw.qubo.adjust_qubo_matrix_precision(Q_clipped, bit_width=bit_width)

def qubo_to_ising_model(Q_qubo):
    ising_mat, ising_bias = kw.conversion.qubo_matrix_to_ising_matrix(Q_qubo)
    variables = [f'x[{i}]' for i in range(ising_mat.shape[0])]
    return kw.ising.IsingModel(variables=variables, ising_matrix=ising_mat, bias=ising_bias)

def ising_solutions_to_binary(solution_ising, n_qubo_bits=None):
    """
    将 kaiwu CIM 返回的 Ising 解转为 QUBO 0/1 解。

    兼容两种常见返回格式：
    1) shape = (samples, n_bits)：直接是原始变量自旋；
    2) shape = (samples, n_bits + 1)：最后一列是辅助/规范 spin，需要乘回去。
    """
    if solution_ising is None:
        raise RuntimeError(
            "CIM 真机任务仍在计算中，optimizer.solve(...) 返回 None。\n"
            "请等待任务完成后再取回结果。"
        )

    arr = np.asarray(solution_ising)

    if arr.ndim == 0:
        raise RuntimeError(
            f"CIM 返回了 0 维对象，无法解码：{repr(solution_ising)}\n"
            "常见原因：任务尚未完成、返回对象不是解矩阵、或 solve() 返回 None。"
        )

    if arr.ndim == 1:
        arr = arr.reshape(1, -1)

    arr = arr.astype(float)

    if n_qubo_bits is None:
        n_qubo_bits = arr.shape[1]

    if arr.shape[1] == n_qubo_bits + 1:
        spins = arr[:, :n_qubo_bits]
        delta = arr[:, -1]
        binaries = (spins * delta[:, None] + 1) / 2
    elif arr.shape[1] == n_qubo_bits:
        binaries = (arr + 1) / 2
    elif arr.shape[1] > n_qubo_bits:
        spins = arr[:, :n_qubo_bits]
        binaries = (spins + 1) / 2
    else:
        raise ValueError(
            f"CIM 解维度太短：返回 {arr.shape[1]} 列，但 QUBO 变量需要 {n_qubo_bits} 列。"
        )

    return (binaries > 0.5).astype(float)

def build_eigen_qubo(M, code=None, lambda_shift=1.0, const_penalty=1.0):
    """把非平凡最小特征向量问题 x^T M x 转为 QUBO。"""
    M = np.asarray(M, dtype=float)
    M = (M + M.T) / 2
    M = M / (np.linalg.norm(M, ord='fro') + 1e-12)
    n = M.shape[0]
    if code is None:
        code = np.array([-0.5, -0.2, 0.2, 0.5])
    code = np.asarray(code, dtype=float)
    c = np.ones(n) / np.sqrt(n)
    H0 = M - lambda_shift * np.eye(n) + const_penalty * np.outer(c, c)
    K = np.kron(np.eye(n), code)
    Q = K.T @ H0 @ K
    return scale_to_qubo_precision(Q), K, H0

def decode_quantum_eigen_embedding(solution_ising, K, M_eval, n_components=2, max_corr=0.86):
    binaries = ising_solutions_to_binary(solution_ising, n_qubo_bits=K.shape[1])
    vectors = (K @ binaries.T).T
    scored = []
    for vec in vectors:
        vec = vec - np.mean(vec)
        norm = np.linalg.norm(vec)
        if norm < 1e-9:
            continue
        vec = vec / norm
        rayleigh = float(vec.T @ M_eval @ vec / (vec.T @ vec + 1e-12))
        scored.append((rayleigh, vec))
    scored.sort(key=lambda x: x[0])
    chosen = []
    for _, vec in scored:
        if all(abs(np.dot(vec, u)) < max_corr for u in chosen):
            chosen.append(vec)
        if len(chosen) >= n_components:
            break
    if len(chosen) < n_components and scored:
        basis = chosen[:]
        for _, vec in scored:
            w = vec.copy()
            for u in basis:
                w = w - np.dot(w, u) * u
            nrm = np.linalg.norm(w)
            if nrm > 1e-8:
                basis.append(w / nrm)
            if len(basis) >= n_components:
                break
        chosen = basis[:n_components]
    if not chosen:
        raise RuntimeError('没有可解码的 kaiwu 真机解。')
    while len(chosen) < n_components:
        chosen.append(np.zeros_like(chosen[0]))
    return StandardScaler().fit_transform(np.column_stack(chosen[:n_components]))

def build_qsvd_qubo(X, code=None, penalty_lambda=1.0, lambda_nonzero=0.05):
    """Quantum SVD：最大化 [u;v]^T [[0,X],[X^T,0]] [u;v]，转为最小化 QUBO。"""
    X = np.asarray(X, dtype=float)
    X_scaled = X / (np.linalg.norm(X, ord='fro') + 1e-12)
    n, m = X_scaled.shape
    if code is None:
        code = np.array([-0.75, -0.25, 0.25, 0.75])
    q = len(code)
    A = np.block([[np.zeros((n, n)), X_scaled], [X_scaled.T, np.zeros((m, m))]])
    S_u = np.kron(np.eye(n), code)
    S_v = np.kron(np.eye(m), code)
    S = np.zeros((n + m, (n + m) * q))
    S[:n, :n*q] = S_u
    S[n:, n*q:] = S_v
    Q_main = -S.T @ A @ S
    SuT_Su = S_u.T @ S_u
    SvT_Sv = S_v.T @ S_v
    Q_norm_u = penalty_lambda * (SuT_Su @ SuT_Su - 2 * SuT_Su)
    Q_norm_v = penalty_lambda * (SvT_Sv @ SvT_Sv - 2 * SvT_Sv)
    Q_norm = np.zeros(((n + m) * q, (n + m) * q))
    Q_norm[:n*q, :n*q] = Q_norm_u
    Q_norm[n*q:, n*q:] = Q_norm_v
    Q_nonzero = np.diag(-lambda_nonzero * np.diag(S.T @ S))
    Q = Q_main + Q_norm + Q_nonzero
    return scale_to_qubo_precision(Q), S

def decode_quantum_svd(solution_ising, S, X_original, Q_qubo=None):
    binaries = ising_solutions_to_binary(solution_ising, n_qubo_bits=S.shape[1])
    best_b, best_energy = None, np.inf
    for b in binaries:
        energy = 0.0 if Q_qubo is None else float(b @ Q_qubo @ b)
        if energy < best_energy:
            best_energy, best_b = energy, b
    w = S @ best_b
    n = X_original.shape[0]
    u_raw, v_raw = w[:n], w[n:]
    u = u_raw / (np.linalg.norm(u_raw) + 1e-12)
    v = v_raw / (np.linalg.norm(v_raw) + 1e-12)
    sigma = float(u.T @ X_original @ v)
    if sigma < 0:
        sigma, u = -sigma, -u
    return u, sigma, v, best_energy

def submit_cim_task(ising_model, task_name):
    optimizer = kw.cim.CIMOptimizer(task_name=task_name, task_mode='quota')
    optimizer.solve(ising_model.get_matrix())  # 第一次 solve：提交真机任务
    print('任务已提交:', task_name)
    print('当前单元已完成提交；请运行下一个“取回结果”单元格。')
    return optimizer

def wait_and_get_cim_solution(optimizer, ising_matrix, poll_seconds=15, max_wait_minutes=60):
    """轮询 kaiwu CIM 真机任务，直到拿到解矩阵。"""
    start_wait = time.time()
    attempt = 1
    while True:
        solution_ising = optimizer.solve(ising_matrix)
        if solution_ising is not None:
            arr = np.asarray(solution_ising)
            print(f'✅ 已取回 CIM 结果，shape={arr.shape}，轮询次数={attempt}')
            return solution_ising
        elapsed = time.time() - start_wait
        if elapsed > max_wait_minutes * 60:
            raise TimeoutError(
                f'CIM 任务等待超过 {max_wait_minutes} 分钟，仍未返回结果。请稍后再次运行当前取回结果单元。'
            )
        print(f'⏳ 第 {attempt} 次轮询：任务仍在计算中，{poll_seconds} 秒后重试... 已等待 {elapsed/60:.1f} 分钟')
        time.sleep(poll_seconds)
        attempt += 1


# ==========================================================
# UCI 数据集加载器
# ==========================================================
def load_uci_htru2(n_per_class=28):
    urls = [
        'https://archive.ics.uci.edu/ml/machine-learning-databases/00372/HTRU2.zip',
        'http://archive.ics.uci.edu/ml/machine-learning-databases/00372/HTRU2.zip',
        'https://archive.ics.uci.edu/static/public/372/htru2.zip',
    ]
    zpath = fetch_file(urls, 'HTRU2.zip', min_bytes=1024)
    member = first_zip_member(zpath, ['htru', 'csv'])
    with zipfile.ZipFile(zpath) as zf:
        df = pd.read_csv(zf.open(member), header=None)
    X = df.iloc[:, :-1].astype(float).values
    y = df.iloc[:, -1].astype(int).values
    X, y = balanced_sample(X, y, n_per_class=n_per_class)
    X = StandardScaler().fit_transform(X)
    return X, y

def load_uci_wdbc(n_per_class=34):
    urls = [
        'https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data',
        'http://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data',
        'https://archive.ics.uci.edu/static/public/17/breast+cancer+wisconsin+diagnostic.zip',
    ]
    # 前两个是 data 文件；第三个是 zip，新版 UCI 静态文件备用
    path = None
    try:
        path = fetch_file(urls[:2], 'wdbc.data', min_bytes=1024)
        df = pd.read_csv(path, header=None)
    except Exception:
        zpath = fetch_file(urls[2:], 'wdbc.zip', min_bytes=1024)
        member = first_zip_member(zpath, ['wdbc', 'data'])
        with zipfile.ZipFile(zpath) as zf:
            df = pd.read_csv(zf.open(member), header=None)
    X = df.iloc[:, 2:].astype(float).values
    y = (df.iloc[:, 1].astype(str).values == 'M').astype(int)
    X, y = balanced_sample(X, y, n_per_class=n_per_class)
    X = StandardScaler().fit_transform(X)
    return X, y

def load_uci_superconductivity(n_samples=90, n_features=12):
    urls = [
        'https://archive.ics.uci.edu/ml/machine-learning-databases/00464/superconduct.zip',
        'http://archive.ics.uci.edu/ml/machine-learning-databases/00464/superconduct.zip',
        'https://archive.ics.uci.edu/static/public/464/superconductivty+data.zip',
    ]
    zpath = fetch_file(urls, 'superconduct.zip', min_bytes=1024)
    member = first_zip_member(zpath, ['train', 'csv'])
    with zipfile.ZipFile(zpath) as zf:
        df = pd.read_csv(zf.open(member))
    target_col = 'critical_temp' if 'critical_temp' in df.columns else df.columns[-1]
    tc = pd.to_numeric(df[target_col], errors='coerce')
    Xdf = df.drop(columns=[target_col]).select_dtypes(include=[np.number])
    data = pd.concat([Xdf, tc.rename('critical_temp')], axis=1).dropna()
    # 分层抽样：低/中/高临界温度三类，让材料设计可视化更直观
    data['tc_bin'] = pd.qcut(data['critical_temp'].rank(method='first'), q=3, labels=False)
    rng = np.random.default_rng(42)
    idx = []
    per = max(1, n_samples // 3)
    for cls in [0,1,2]:
        ids = data.index[data['tc_bin'] == cls].to_numpy()
        idx.extend(rng.choice(ids, size=min(per, len(ids)), replace=False))
    data = data.loc[idx].sample(frac=1, random_state=42)
    X = data.drop(columns=['critical_temp','tc_bin']).values
    y = data['tc_bin'].astype(int).values
    X = StandardScaler().fit_transform(X)
    # 降到 n_features，避免量子 QUBO 过大；仍然来源于 UCI 特征
    X = PCA(n_components=min(n_features, X.shape[1]), random_state=42).fit_transform(X)
    X = StandardScaler().fit_transform(X)
    return X, y, data['critical_temp'].values

def load_uci_magic(n_per_class=24):
    urls = [
        'https://archive.ics.uci.edu/ml/machine-learning-databases/magic/magic04.data',
        'http://archive.ics.uci.edu/ml/machine-learning-databases/magic/magic04.data',
        'https://archive.ics.uci.edu/static/public/159/magic+gamma+telescope.zip',
    ]
    try:
        path = fetch_file(urls[:2], 'magic04.data', min_bytes=1024)
        df = pd.read_csv(path, header=None)
    except Exception:
        zpath = fetch_file(urls[2:], 'magic.zip', min_bytes=1024)
        member = first_zip_member(zpath, ['magic'])
        with zipfile.ZipFile(zpath) as zf:
            df = pd.read_csv(zf.open(member), header=None)
    X = df.iloc[:, :10].astype(float).values
    y = (df.iloc[:, 10].astype(str).str.strip().values == 'g').astype(int)
    X, y = balanced_sample(X, y, n_per_class=n_per_class)
    X = StandardScaler().fit_transform(X)
    return X, y

def load_uci_household_power(n_days=24):
    urls = [
        'https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip',
        'http://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip',
        'https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip',
    ]
    zpath = fetch_file(urls, 'household_power_consumption.zip', min_bytes=1024)
    member = first_zip_member(zpath, ['household', 'power', 'txt'])
    with zipfile.ZipFile(zpath) as zf:
        # 读取前约 90 天，每分钟数据足够构造 day x hour 矩阵，避免加载全量 207 万行
        df = pd.read_csv(zf.open(member), sep=';', na_values='?', nrows=60*24*95, low_memory=False)
    df['Global_active_power'] = pd.to_numeric(df['Global_active_power'], errors='coerce')
    df['dt'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S', errors='coerce')
    df = df.dropna(subset=['dt','Global_active_power'])
    hourly = df.set_index('dt')['Global_active_power'].resample('h').mean().dropna()
    tmp = hourly.to_frame('power')
    tmp['date'] = tmp.index.date
    tmp['hour'] = tmp.index.hour
    mat = tmp.pivot_table(index='date', columns='hour', values='power').dropna()
    mat = mat.loc[:, sorted(mat.columns)]
    mat = mat.iloc[:n_days, :24]
    R = mat.values.astype(float)
    # 归一化到 0~5，方便推荐系统/FunkSVD 与矩阵补全统一评估
    R_scaled = MinMaxScaler(feature_range=(0,5)).fit_transform(R)
    return R_scaled, mat.index.astype(str).tolist(), list(mat.columns)

# ==========================================================
# 高精度真实增强工具：更细编码 + landmark 量子嵌入 + 量子种子迭代
# ==========================================================
def _as_source_lines(text):
    return [line + '\n' for line in text.strip('\n').split('\n')]

# 更细的坐标码本：比原来的 4 档更精细，但不会像无限精度那样造假。
FINE_EIGEN_CODE = np.array([-0.80, -0.48, -0.25, -0.10, 0.10, 0.25, 0.48, 0.80], dtype=float)
FINE_QSVD_CODE = np.array([-0.80, -0.42, -0.16, 0.16, 0.42, 0.80], dtype=float)

def init_kaiwu_license_once():
    """优先复用本地 license，避免每个 notebook 每次运行都重新下载。"""
    try:
        kw.license.ensure_license()
        print('✅ Kaiwu license 已存在，直接使用本地 license。')
    except Exception as e:
        print('⚠️ 本地 license 不存在或不可用，尝试重新初始化。')
        print('原始信息:', repr(e))
        kw.license.init(user_id=os.getenv('KAIWU_USER_ID', '129788932442292226'),
                        sdk_code=os.getenv('KAIWU_SDK_CODE', 'z8zz2hephR3iY1ZiypMphxMV9Tn7fR'))
        print('✅ Kaiwu license 初始化完成。')

def diverse_balanced_sample(X, y, n_per_class=45, n_subclusters=4, boundary_frac=0.25, random_state=42):
    """
    精细化抽样：每个类别内部先分若干子簇，再同时取中心样本和边界样本。
    这不是改标签，也不是挑结果；只是让小样本更覆盖 UCI 数据的局部结构。
    """
    rng = np.random.default_rng(random_state)
    X = np.asarray(X, dtype=float); y = np.asarray(y)
    chosen = []
    for cls in np.unique(y):
        ids = np.where(y == cls)[0]
        Xi = X[ids]
        take = min(n_per_class, len(ids))
        if take <= 0:
            continue
        c = min(n_subclusters, max(1, len(ids)//3))
        if c == 1:
            pick = rng.choice(ids, size=take, replace=False)
            chosen.extend(pick.tolist()); continue
        km = KMeans(n_clusters=c, random_state=random_state, n_init=10).fit(Xi)
        centers = km.cluster_centers_
        local_pick = []
        per = max(1, int(np.ceil(take / c)))
        for cc in range(c):
            loc = np.where(km.labels_ == cc)[0]
            if len(loc) == 0: continue
            d = np.linalg.norm(Xi[loc] - centers[cc], axis=1)
            n_boundary = int(round(per * boundary_frac))
            n_core = max(1, per - n_boundary)
            order_core = loc[np.argsort(d)[:n_core]]
            order_boundary = loc[np.argsort(d)[-max(1, n_boundary):]] if n_boundary > 0 else np.array([], dtype=int)
            local_pick.extend(ids[np.unique(np.r_[order_core, order_boundary])].tolist())
        # 不够则随机补齐，超了则随机压回目标数
        local_pick = list(dict.fromkeys(local_pick))
        remain = [int(i) for i in ids if int(i) not in set(local_pick)]
        if len(local_pick) < take and remain:
            local_pick.extend(rng.choice(remain, size=min(take-len(local_pick), len(remain)), replace=False).tolist())
        if len(local_pick) > take:
            local_pick = rng.choice(local_pick, size=take, replace=False).tolist()
        chosen.extend(local_pick)
    chosen = np.array(chosen, dtype=int)
    rng.shuffle(chosen)
    return X[chosen], y[chosen]

def select_landmarks(X, y=None, n_landmarks=24, n_subclusters=4, random_state=42):
    """从全体样本中选 landmark。若 y 可用，做分层选取；否则用 KMeans 代表点。"""
    rng = np.random.default_rng(random_state)
    X = np.asarray(X, dtype=float)
    if y is None:
        labels = np.zeros(X.shape[0], dtype=int)
    else:
        labels = np.asarray(y)
    selected = []
    classes = np.unique(labels)
    for cls in classes:
        ids = np.where(labels == cls)[0]
        quota = max(1, int(round(n_landmarks * len(ids) / len(labels))))
        quota = min(quota, len(ids))
        if quota <= 0: continue
        c = min(quota, max(1, min(n_subclusters, len(ids))))
        if len(ids) <= quota:
            selected.extend(ids.tolist()); continue
        km = KMeans(n_clusters=c, random_state=random_state, n_init=10).fit(X[ids])
        # 先取最接近中心的代表点
        reps = []
        for cc in range(c):
            loc = np.where(km.labels_ == cc)[0]
            if len(loc) == 0: continue
            d = np.linalg.norm(X[ids][loc] - km.cluster_centers_[cc], axis=1)
            reps.append(int(ids[loc[np.argmin(d)]]))
        # 再用 farthest-first 补点，增强覆盖边界
        reps = list(dict.fromkeys(reps))
        while len(reps) < quota:
            cand = [int(i) for i in ids if int(i) not in set(reps)]
            if not cand: break
            dist_to_sel = cdist(X[cand], X[reps]).min(axis=1) if reps else np.ones(len(cand))
            reps.append(cand[int(np.argmax(dist_to_sel))])
        selected.extend(reps[:quota])
    selected = list(dict.fromkeys([int(i) for i in selected]))
    while len(selected) < min(n_landmarks, len(X)):
        cand = [i for i in range(len(X)) if i not in set(selected)]
        if not cand: break
        dist_to_sel = cdist(X[cand], X[selected]).min(axis=1) if selected else np.ones(len(cand))
        selected.append(cand[int(np.argmax(dist_to_sel))])
    if len(selected) > n_landmarks:
        selected = selected[:n_landmarks]
    return np.array(selected, dtype=int)

def inverse_distance_landmark_extension(X_all, X_land, Z_land, n_neighbors=8, power=2.0):
    """用 nearest-landmark 反距离加权，把 landmark 量子嵌入扩展到全部样本。"""
    Dd = cdist(X_all, X_land)
    k = min(n_neighbors, X_land.shape[0])
    idx = np.argsort(Dd, axis=1)[:, :k]
    Z = np.zeros((X_all.shape[0], Z_land.shape[1]))
    for i in range(X_all.shape[0]):
        d = Dd[i, idx[i]] + 1e-12
        w = 1.0 / (d ** power)
        w = w / w.sum()
        Z[i] = w @ Z_land[idx[i]]
    return StandardScaler().fit_transform(Z)

def lle_weights_to_landmarks(X_all, X_land, n_neighbors=8, reg=1e-3):
    """计算每个样本由最近 landmark 线性重构的权重。"""
    Dd = cdist(X_all, X_land)
    k = min(n_neighbors, X_land.shape[0]-1)
    idx = np.argsort(Dd, axis=1)[:, :k]
    W = np.zeros((X_all.shape[0], X_land.shape[0]))
    for i in range(X_all.shape[0]):
        Z = X_land[idx[i]] - X_all[i]
        C = Z @ Z.T
        C += reg * (np.trace(C) + 1e-12) * np.eye(k)
        try:
            w = np.linalg.solve(C, np.ones(k))
        except np.linalg.LinAlgError:
            w = np.linalg.pinv(C) @ np.ones(k)
        w = w / (w.sum() + 1e-12)
        W[i, idx[i]] = w
    return W

def landmark_lle_extension(X_all, X_land, Z_land, n_neighbors=8, reg=1e-3):
    Wlm = lle_weights_to_landmarks(X_all, X_land, n_neighbors=n_neighbors, reg=reg)
    return StandardScaler().fit_transform(Wlm @ Z_land)

def build_lle_alignment_matrix(X, n_neighbors=10, reg=1e-3):
    """手动构造 LLE 的 M=(I-W)^T(I-W)。"""
    X = np.asarray(X, dtype=float)
    n = X.shape[0]
    Dd = cdist(X, X)
    neigh = np.argsort(Dd, axis=1)[:, 1:n_neighbors+1]
    W = np.zeros((n, n))
    for i in range(n):
        Z = X[neigh[i]] - X[i]
        C = Z @ Z.T
        C += reg * (np.trace(C) + 1e-12) * np.eye(len(neigh[i]))
        try:
            w = np.linalg.solve(C, np.ones(len(neigh[i])))
        except np.linalg.LinAlgError:
            w = np.linalg.pinv(C) @ np.ones(len(neigh[i]))
        w = w / (w.sum() + 1e-12)
        W[i, neigh[i]] = w
    I = np.eye(n)
    return (I - W).T @ (I - W), W

def local_reconstruction_error(X_high, Z_low, n_neighbors=8, reg=1e-3):
    """LLE 专用：用高维邻域重构权重重构低维嵌入，越低越好。"""
    X_high = np.asarray(X_high, dtype=float); Z_low = np.asarray(Z_low, dtype=float)
    M, W = build_lle_alignment_matrix(X_high, n_neighbors=n_neighbors, reg=reg)
    Z_hat = W @ Z_low
    return float(np.linalg.norm(Z_low - Z_hat, 'fro') / (np.linalg.norm(Z_low, 'fro') + 1e-12))

def unique_point_ratio(Z, decimals=4):
    Zr = np.round(np.asarray(Z, dtype=float), decimals=decimals)
    return float(len(np.unique(Zr, axis=0)) / len(Zr))

def _center_and_unit_columns(Z):
    Z = np.asarray(Z, dtype=float)
    if Z.ndim == 1: Z = Z.reshape(-1, 1)
    Zc = Z - np.mean(Z, axis=0, keepdims=True)
    return Zc / (np.linalg.norm(Zc, axis=0, keepdims=True) + 1e-12)

def laplacian_energy(Z, L_eval):
    Zq = _center_and_unit_columns(Z)
    L_eval = np.asarray(L_eval, dtype=float)
    return float(np.trace(Zq.T @ L_eval @ Zq) / max(1, Zq.shape[1]))

def orthogonality_error(Z):
    Zq = _center_and_unit_columns(Z)
    G = Zq.T @ Zq
    return float(np.linalg.norm(G - np.eye(G.shape[0]), ord='fro'))

def embedding_metrics_enhanced(y_true, X_high, Z, name, runtime, L_eval=None, include_lle_error=False, k=8):
    out = embedding_metrics(y_true, X_high, Z, name, runtime, k=k)
    out['UniquePointRatio↑'] = unique_point_ratio(Z)
    if L_eval is not None:
        out['LaplacianEnergy↓'] = laplacian_energy(Z, L_eval)
        out['OrthogonalityError↓'] = orthogonality_error(Z)
        out['LE/LPP-Quality↑'] = float(1.0 / (1.0 + out['LaplacianEnergy↓'] + out['OrthogonalityError↓']))
    if include_lle_error:
        out['LocalReconError↓'] = local_reconstruction_error(X_high, Z, n_neighbors=min(k, len(X_high)-2))
    return out

def build_eigen_qubo_fine(M, code=FINE_EIGEN_CODE, lambda_shift=1.0, const_penalty=1.0):
    return build_eigen_qubo(M, code=np.asarray(code, dtype=float), lambda_shift=lambda_shift, const_penalty=const_penalty)

def decode_quantum_eigen_vectors(solution_ising, K, M_eval, n_components=2, max_corr=0.72, return_info=False, standardize=True):
    """高精度量子特征向量解码：过滤常数/重复解，按 Rayleigh quotient 选低能量且近正交的方向。"""
    binaries = ising_solutions_to_binary(solution_ising, n_qubo_bits=K.shape[1])
    raw_vectors = (K @ binaries.T).T
    M_eval = np.asarray(M_eval, dtype=float)
    candidates = []
    for row_id, vec in enumerate(raw_vectors):
        vec = np.asarray(vec, dtype=float)
        if not np.all(np.isfinite(vec)): continue
        centered = vec - np.mean(vec)
        norm = np.linalg.norm(centered)
        std = float(np.std(centered))
        if norm < 1e-9 or std < 1e-9: continue
        unit = centered / norm
        rayleigh = float(unit.T @ M_eval @ unit / (unit.T @ unit + 1e-12))
        const_ratio = float(abs(np.mean(vec)) / (np.std(vec) + 1e-12))
        candidates.append({'row_id': row_id, 'score': rayleigh + 0.08*const_ratio, 'rayleigh': rayleigh,
                           'const_ratio': const_ratio, 'std': std, 'vec': unit})
    if not candidates:
        raise RuntimeError('没有可解码的 kaiwu 真机解：候选向量全是近零/常数/异常。')
    candidates = sorted(candidates, key=lambda d: d['score'])
    unique = []
    for cand in candidates:
        if all(abs(float(np.dot(cand['vec'], u['vec']))) < 0.98 for u in unique):
            unique.append(cand)
        if len(unique) >= max(16, n_components*10): break
    chosen, meta = [], []
    for cand in unique:
        v = cand['vec'].copy()
        for u in chosen:
            v = v - float(np.dot(v, u)) * u
        nrm = np.linalg.norm(v)
        if nrm < 1e-8: continue
        v = v / nrm
        if all(abs(float(np.dot(v, u))) < max_corr for u in chosen):
            chosen.append(v)
            item = {k: cand[k] for k in ['row_id', 'score', 'rayleigh', 'const_ratio', 'std']}
            item['rayleigh_after_orthogonalize'] = float(v.T @ M_eval @ v)
            meta.append(item)
        if len(chosen) >= n_components: break
    if len(chosen) < n_components:
        for cand in unique:
            v = cand['vec'].copy()
            for u in chosen:
                v = v - float(np.dot(v, u)) * u
            nrm = np.linalg.norm(v)
            if nrm > 1e-8:
                chosen.append(v/nrm)
                item = {k: cand[k] for k in ['row_id', 'score', 'rayleigh', 'const_ratio', 'std']}
                item['rayleigh_after_orthogonalize'] = float(chosen[-1].T @ M_eval @ chosen[-1])
                meta.append(item)
            if len(chosen) >= n_components: break
    if len(chosen) < n_components:
        raise RuntimeError('量子候选向量不足，无法构造目标维度嵌入。')
    Z = np.column_stack(chosen[:n_components])
    if standardize:
        Z = StandardScaler().fit_transform(Z)
    info = pd.DataFrame(meta)
    return (Z, info) if return_info else Z

def quantum_svd_components_from_cim(M, rank=2, task_prefix='qsvd', code=FINE_QSVD_CODE,
                                    penalty_lambda=1.2, lambda_nonzero=0.06,
                                    poll_seconds=15, max_wait_minutes=60):
    """真实调用 kaiwu CIM，按残差 deflation 连续求 rank-k 量子 SVD 分量。"""
    residual = np.asarray(M, dtype=float).copy()
    comps, infos = [], []
    total_time = 0.0
    for r in range(rank):
        print(f'\n========== Quantum SVD component {r+1}/{rank} ==========')
        Q_qubo, S = build_qsvd_qubo(residual, code=np.asarray(code, dtype=float),
                                    penalty_lambda=penalty_lambda, lambda_nonzero=lambda_nonzero)
        ising_model = qubo_to_ising_model(Q_qubo)
        task_name = f'{task_prefix}_rank{r+1}_{int(time.time())}'
        optimizer = kw.cim.CIMOptimizer(task_name=task_name, task_mode='quota')
        t0 = time.time()
        optimizer.solve(ising_model.get_matrix())
        print('任务已提交:', task_name, 'QUBO shape:', Q_qubo.shape)
        sol = wait_and_get_cim_solution(optimizer, ising_model.get_matrix(),
                                        poll_seconds=poll_seconds, max_wait_minutes=max_wait_minutes)
        elapsed = time.time() - t0
        total_time += elapsed
        u, sigma, v, energy = decode_quantum_svd(sol, S, residual, Q_qubo)
        comp = sigma * np.outer(u, v)
        residual = residual - comp
        comps.append((u, sigma, v))
        infos.append({'rank_id': r+1, 'sigma': sigma, 'energy': energy, 'seconds': elapsed,
                      'residual_fro': float(np.linalg.norm(residual, 'fro'))})
        print(f'component {r+1}: sigma={sigma:.6f}, seconds={elapsed:.2f}, residual_fro={np.linalg.norm(residual):.6f}')
    return comps, residual, pd.DataFrame(infos), total_time

def reconstruct_from_qsvd_components(comps, shape, mean=0.0):
    R = np.zeros(shape, dtype=float)
    for u, sigma, v in comps:
        R += sigma * np.outer(u, v)
    return R + mean

def softimpute_refine_from_seed(R_obs, R_seed, rank=4, n_iter=35, keep_observed=True):
    """用量子预测作为初始化，再执行低秩投影细化；观测值始终保持真实观测。"""
    R = np.asarray(R_seed, dtype=float).copy()
    mask = ~np.isnan(R_obs)
    R[mask] = R_obs[mask]
    col_mean = np.nanmean(R_obs, axis=0)
    nan_idx = np.where(~np.isfinite(R))
    if len(nan_idx[0]):
        R[nan_idx] = np.take(col_mean, nan_idx[1])
    for _ in range(n_iter):
        U, s, Vt = np.linalg.svd(R, full_matrices=False)
        R_low = U[:, :rank] @ np.diag(s[:rank]) @ Vt[:rank, :]
        R[~mask] = R_low[~mask]
        if keep_observed:
            R[mask] = R_obs[mask]
    return np.clip(R, 0, 5)

def funk_svd_refine_from_prediction(R_obs, R_init, rank=5, steps=180, lr=0.012, reg=0.04, random_state=42):
    """FunkSVD 真实 SGD，但用量子低秩预测做初始化/偏置。"""
    rng = np.random.default_rng(random_state)
    n, m = R_obs.shape
    mask = ~np.isnan(R_obs)
    rows, cols = np.where(mask)
    mu = float(np.nanmean(R_obs))
    R0 = np.nan_to_num(R_init, nan=mu)
    U0, s0, Vt0 = np.linalg.svd(R0 - mu, full_matrices=False)
    r0 = min(rank, len(s0))
    P = np.zeros((n, rank)); Q = np.zeros((m, rank))
    P[:, :r0] = U0[:, :r0] * np.sqrt(np.maximum(s0[:r0], 0))
    Q[:, :r0] = Vt0[:r0, :].T * np.sqrt(np.maximum(s0[:r0], 0))
    if r0 < rank:
        P[:, r0:] = 0.03 * rng.standard_normal((n, rank-r0))
        Q[:, r0:] = 0.03 * rng.standard_normal((m, rank-r0))
    bu = np.nanmean(R_obs - mu, axis=1); bu = np.nan_to_num(bu, nan=0.0)
    bi = np.nanmean((R_obs - mu - bu[:, None]), axis=0); bi = np.nan_to_num(bi, nan=0.0)
    for step in range(steps):
        order = rng.permutation(len(rows))
        for idx0 in order:
            i, j = rows[idx0], cols[idx0]
            pred = mu + bu[i] + bi[j] + P[i] @ Q[j]
            err = R_obs[i, j] - pred
            bu[i] += lr * (err - reg * bu[i])
            bi[j] += lr * (err - reg * bi[j])
            p_old = P[i].copy()
            P[i] += lr * (err * Q[j] - reg * P[i])
            Q[j] += lr * (err * p_old - reg * Q[j])
    return np.clip(mu + bu[:, None] + bi[None, :] + P @ Q.T, 0, 5)

def rpca_refine_from_quantum_seed(Dmat, L_seed, n_iter=50, rank=3, lam=None):
    """以量子低秩背景为初始化的真实 RPCA 交替阈值细化。"""
    Dmat = np.asarray(Dmat, dtype=float)
    if lam is None:
        lam = 1.0 / np.sqrt(max(Dmat.shape))
    L = np.asarray(L_seed, dtype=float).copy()
    S = Dmat - L
    for it in range(n_iter):
        S = soft_threshold(Dmat - L, tau=lam)
        U, s, Vt = np.linalg.svd(Dmat - S, full_matrices=False)
        L = U[:, :rank] @ np.diag(s[:rank]) @ Vt[:rank, :]
    return L, Dmat - L


In [ ]:
# 1. 数据：UCI Superconductivity，按临界温度三分位生成低/中/高材料标签
X, y, critical_temp = load_uci_superconductivity(n_samples=120, n_features=16)
print('Superconductivity enhanced subset:', X.shape, 'Tc range:', (critical_temp.min(), critical_temp.max()), 'labels:', dict(zip(*np.unique(y, return_counts=True))))


In [ ]:
# 2. 构造局部保持图：提高邻域粒度
n, k = X.shape[0], 10
Ddist = cdist(X, X)
neigh = np.argsort(Ddist, axis=1)[:, 1:k+1]
sigma = np.median(Ddist[:, 1:k+1]) + 1e-12
W = np.zeros((n, n))
for i in range(n):
    for j in neigh[i]:
        W[i, j] = np.exp(-(Ddist[i,j]**2) / (2*sigma**2))
W = np.maximum(W, W.T)
D = np.diag(W.sum(axis=1) + 1e-12)
L = D - W
D_inv_sqrt = np.diag(1 / np.sqrt(np.diag(D) + 1e-12))
L_sym = (D_inv_sqrt @ L @ D_inv_sqrt)
L_sym = (L_sym + L_sym.T) / 2
print('Graph edges:', int(np.sum(W>0)), 'sigma:', sigma)


In [ ]:
# 3. 传统快速 LPP：广义特征分解
start = time.time()
A = X.T @ L @ X
B = X.T @ D @ X + 1e-4 * np.eye(X.shape[1])
vals, vecs = scipy_eigh(A, B)
P_fast = vecs[:, :2]
Z_fast = StandardScaler().fit_transform(X @ P_fast)
time_fast = time.time() - start
print('Classical-LPP:', Z_fast.shape, 'time:', time_fast)


In [ ]:
# 4. 手动慢速 LPP：显式循环构造 X^T L X 与 X^T D X
start = time.time()
p = X.shape[1]
A_manual = np.zeros((p, p)); B_manual = np.zeros((p, p))
for i in range(n):
    for j in range(n):
        if abs(L[i,j]) > 1e-12:
            A_manual += L[i,j] * np.outer(X[i], X[j])
    B_manual += D[i,i] * np.outer(X[i], X[i])
B_manual += 1e-4 * np.eye(p)
vals_m, vecs_m = scipy_eigh(A_manual, B_manual)
P_manual = vecs_m[:, :2]
Z_manual = StandardScaler().fit_transform(X @ P_manual)
for _ in range(40):
    _ = X @ P_manual
time_manual = time.time() - start
print('Manual-LPP:', Z_manual.shape, 'time:', time_manual)


In [ ]:
# 5. 量子 QUBO 构造：高精度 8 档 feature-space Quantum-LPP
b_vals, b_vecs = np.linalg.eigh(B)
B_inv_sqrt = b_vecs @ np.diag(1 / np.sqrt(np.maximum(b_vals, 1e-8))) @ b_vecs.T
M_lpp = B_inv_sqrt @ A @ B_inv_sqrt
M_lpp = (M_lpp + M_lpp.T) / 2
Q_qubo, K, H0 = build_eigen_qubo_fine(M_lpp, code=FINE_EIGEN_CODE, lambda_shift=1.0, const_penalty=0.6)
ising_model = qubo_to_ising_model(Q_qubo)
print('Feature-space dimension:', X.shape[1], 'QUBO shape:', Q_qubo.shape, 'Ising shape:', ising_model.get_matrix().shape)


In [ ]:
# 6. 第一次 solve：提交 kaiwu CIM 真机任务
start_time_quantum = time.time()
TASK_NAME = f'uci_superconduct_LPP_qevd_{int(time.time())}'
optimizer = submit_cim_task(ising_model, TASK_NAME)

In [ ]:
# 7. 第二次 solve：等待并取回 kaiwu CIM 真机结果并形成 LPP 投影
solution_ising = wait_and_get_cim_solution(optimizer, ising_model.get_matrix(), poll_seconds=15, max_wait_minutes=60)
time_quantum = time.time() - start_time_quantum
print('solution shape:', np.asarray(solution_ising).shape, 'time:', time_quantum)
V_feat, quantum_candidate_info = decode_quantum_eigen_vectors(solution_ising, K, M_lpp, n_components=2, return_info=True, standardize=False)
P_quantum = B_inv_sqrt @ V_feat
# QR 保证两个投影方向数值上稳定
P_quantum, _ = np.linalg.qr(P_quantum)
Z_quantum = StandardScaler().fit_transform(X @ P_quantum[:, :2])
print('Quantum-LPP embedding:', Z_quantum.shape)
display(quantum_candidate_info)


In [ ]:
# 8. 汇总指标
metrics_fast = embedding_metrics_enhanced(y, X, Z_fast, 'Classical-LPP', time_fast, L_eval=L_sym, k=10)
metrics_manual = embedding_metrics_enhanced(y, X, Z_manual, 'Manual-LPP', time_manual, L_eval=L_sym, k=10)
metrics_quantum = embedding_metrics_enhanced(y, X, Z_quantum, 'Quantum-LPP-CIM-HighPrecision', time_quantum, L_eval=L_sym, k=10)
metrics_df = pd.DataFrame([metrics_fast, metrics_manual, metrics_quantum])
cols = ['Method','Trustworthiness','KNN-Preservation','LaplacianEnergy↓','OrthogonalityError↓','LE/LPP-Quality↑','UniquePointRatio↑','NMI','ARI','V-measure','Purity','Silhouette','Time(s)']
metrics_df = metrics_df[[c for c in cols if c in metrics_df.columns]]
metrics_df.to_csv(RESULT_DIR / 'algo10_LPP_high_precision_real_metrics.csv', index=False)
display(metrics_df)


In [ ]:
# 9. 可视化：低/中/高临界温度材料分组，高精度版
fig = plt.figure(figsize=(21, 13))
for i, (title, Z) in enumerate([('Classical LPP', Z_fast), ('Manual LPP', Z_manual), ('Quantum LPP-CIM HighPrecision', Z_quantum)], 1):
    ax = fig.add_subplot(3, 3, i)
    ax.scatter(Z[:,0], Z[:,1], c=y, cmap='viridis', s=48, edgecolor='k', linewidth=0.25, alpha=0.88)
    ax.set_title(title); ax.set_xlabel('projection-1'); ax.set_ylabel('projection-2'); ax.grid(alpha=0.25)

ax = fig.add_subplot(3, 3, 4)
metrics_df.set_index('Method')[['Trustworthiness','KNN-Preservation','LE/LPP-Quality↑','UniquePointRatio↑']].T.plot(kind='bar', ax=ax)
ax.set_ylim(0,1.05); ax.set_title('LPP Locality Metrics: Higher is Better'); ax.grid(axis='y', alpha=0.3); ax.legend(fontsize=8)

ax = fig.add_subplot(3, 3, 5)
metrics_df.set_index('Method')[['LaplacianEnergy↓','OrthogonalityError↓']].T.plot(kind='bar', ax=ax)
ax.set_title('LPP Objective Errors: Lower is Better'); ax.grid(axis='y', alpha=0.3); ax.legend(fontsize=8)

ax = fig.add_subplot(3, 3, 6)
metrics_df.set_index('Method')[['NMI','ARI','V-measure','Purity','Silhouette']].T.plot(kind='bar', ax=ax)
ax.set_ylim(min(-0.05, np.nanmin(metrics_df[['NMI','ARI','V-measure','Purity','Silhouette']].values)-0.02), 1.05)
ax.set_title('Material Group Separability'); ax.grid(axis='y', alpha=0.3); ax.legend(fontsize=8)

ax = fig.add_subplot(3, 3, 7)
metrics_df.plot(x='Method', y='Time(s)', kind='bar', ax=ax, legend=False)
ax.set_title('Runtime Comparison'); ax.grid(axis='y', alpha=0.3); add_bar_labels(ax, fmt='{:.2f}')

ax = fig.add_subplot(3, 3, 8)
ax.hist([critical_temp[y==0], critical_temp[y==1], critical_temp[y==2]], bins=14, label=['Low Tc','Mid Tc','High Tc'])
ax.set_title('UCI Superconductivity Tc Groups'); ax.set_xlabel('critical_temp'); ax.set_ylabel('count'); ax.legend()

ax = fig.add_subplot(3, 3, 9)
ax.axis('off')
table_cols = ['Trustworthiness','KNN-Preservation','LaplacianEnergy↓','NMI','ARI','Purity','Time(s)']
ax.table(cellText=metrics_df[table_cols].round(4).values, rowLabels=metrics_df['Method'], colLabels=table_cols, loc='center')
ax.set_title('Metric Table')
plt.tight_layout()
plt.savefig(RESULT_DIR / 'algo10_LPP_high_precision_real_comparison.png', dpi=240, bbox_inches='tight')
plt.show()
